# LLM Bias Detection - Full Analysis

Complete workflow for detecting demographic bias in language models.

In [ ]:
import sys
sys.path.append('../src')

from experiment import BiasExperiment, PromptGenerator, BiasAnalyzer
from statistical_analysis import StatisticalAnalyzer
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

## Step 1: Generate Test Prompts

In [ ]:
# Create prompt generator
generator = PromptGenerator()

# Generate prompts for gender dimension
prompts = generator.generate_prompts('gender', samples_per_demo=3)

print(f'Generated {len(prompts)} test prompts')
print('\nExample prompts:')
for i, p in enumerate(prompts[:3]):
    print(f"{i+1}. {p['text']}")

## Step 2: Run Experiment

In [ ]:
# Create and run experiment
experiment = BiasExperiment()
experiment.run_experiment('gender', mock_responses=True)

print(f'Analyzed {len(experiment.results)} responses')

## Step 3: Analyze Results

In [ ]:
# Generate report
report = experiment.generate_report('gender')
comparison = report['comparison']

print('Findings:')
for finding in report['findings']:
    print(f'  • {finding}')

## Step 4: Visualize Results

In [ ]:
# Plot sentiment comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment scores
demos = list(comparison.keys())
sentiments = [comparison[d]['avg_sentiment'] for d in demos]

ax1.bar(demos, sentiments, color=['#3498db', '#e74c3c', '#2ecc71'][:len(demos)], edgecolor='black')
ax1.set_ylabel('Average Sentiment')
ax1.set_title('Sentiment by Demographic', fontweight='bold')
ax1.set_ylim(0, 1)
ax1.grid(axis='y', alpha=0.3)

# Bias scores
agentic_scores = [comparison[d]['bias_scores']['agentic'] for d in demos]
communal_scores = [comparison[d]['bias_scores']['communal'] for d in demos]

x = np.arange(len(demos))
width = 0.35

ax2.bar(x - width/2, agentic_scores, width, label='Agentic', color='steelblue')
ax2.bar(x + width/2, communal_scores, width, label='Communal', color='coral')
ax2.set_ylabel('Score')
ax2.set_title('Bias Word Usage', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(demos)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/bias_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 5: Statistical Testing

In [ ]:
# Statistical analysis
analyzer = StatisticalAnalyzer()

# Compare sentiment between groups
if len(demos) >= 2:
    group1_sentiment = [comparison[demos[0]]['avg_sentiment']] * comparison[demos[0]]['n']
    group2_sentiment = [comparison[demos[1]]['avg_sentiment']] * comparison[demos[1]]['n']
    
    result = analyzer.t_test_two_groups(group1_sentiment, group2_sentiment)
    
    print('Statistical Test Results:')
    print(f"  t-statistic: {result['t_statistic']:.3f}")
    print(f"  p-value: {result['p_value']:.4f}")
    print(f"  Significant: {result['significant']}")
    print(f"  Effect size: {result['effect_size']:.3f}")

## Conclusion

This notebook demonstrates:
- Systematic bias testing methodology
- Quantitative analysis of language differences
- Statistical significance testing
- Visualization of results

The framework can be extended to test other dimensions (age, race) and other models.